In [1]:
import os

from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Colab Notebooks/Projek Fluency/Fluensy Model')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import random

print("💸 [FLUENSY AI - THE PRICER] Membangun Model Prediksi Base Rate...\n")

# =========================================================================
# 1. DATA PREPARATION (Dari Cleaned_Rate_Card.csv)
# =========================================================================
file_path = 'Cleaned_Rate_Card.csv'
try:
    df_raw = pd.read_csv(file_path)
except FileNotFoundError:
    print("⚠️ Upload file Cleaned_Rate_Card.csv ke Colab dulu, Bos!")
    raise

# Ambil kolom yang dibutuhkan
df_data = df_raw[['Kategori', 'Followers_IG', 'Rate_Reels_IG']].copy()

# Pastikan tipe data angka
df_data['Followers_IG'] = pd.to_numeric(df_data['Followers_IG'], errors='coerce')
df_data['Rate_Reels_IG'] = pd.to_numeric(df_data['Rate_Reels_IG'], errors='coerce')

# Bersihkan data kosong dan outlier (> Rp 30 Juta)
df_clean = df_data.dropna(subset=['Kategori', 'Followers_IG', 'Rate_Reels_IG']).copy()
df_clean = df_clean[df_clean['Rate_Reels_IG'] <= 30000000]

random.seed(42)
dataset_influencer = []
for _, inf in df_clean.iterrows():
    bio_text = str(inf['Kategori']).replace('\n', ', ')
    followers = int(inf['Followers_IG'])
    rate_reels = int(inf['Rate_Reels_IG'])
    
    # Simulasi ER dan Umur (Keduanya krusial untuk prediksi, namun belum ada di CSV)
    er = round(random.uniform(1.0, 5.0), 2) if followers > 100000 else round(random.uniform(4.0, 9.0), 2)
    i_age = random.randint(18, 35)

    dataset_influencer.append({
        "bio_text": bio_text,
        "followers": followers,
        "er": er,
        "i_age": i_age,
        "target_reels": rate_reels
    })

df_final = pd.DataFrame(dataset_influencer)
# Duplikasi agar AI belajar maksimal
df_final = pd.concat([df_final]*15, ignore_index=True) 

# =========================================================================
# 2. PREPROCESSING
# =========================================================================
scaler_X = MinMaxScaler()
X_num = scaler_X.fit_transform(df_final[['followers', 'er', 'i_age']])

# HANYA 1 TARGET OUTPUT: Rate Reels
scaler_Y = MinMaxScaler()
y_reels = scaler_Y.fit_transform(df_final[['target_reels']])

X_text = df_final['bio_text'].values

X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_text, X_num, y_reels, test_size=0.2, random_state=42
)

# =========================================================================
# 3. CUSTOM LAYER
# =========================================================================
@tf.keras.utils.register_keras_serializable() 
class FluensyAttentionLayer(layers.Layer):
    def __init__(self, **kwargs):
        super(FluensyAttentionLayer, self).__init__(**kwargs)
    def build(self, input_shape):
        self.w = self.add_weight(shape=(input_shape[-1],), initializer='random_normal', trainable=True)
    def call(self, inputs):
        return inputs * tf.nn.sigmoid(self.w)

# =========================================================================
# 4. TENSORFLOW FUNCTIONAL API (SINGLE OUTPUT)
# =========================================================================
max_vocab = 2000
max_len = 30

text_input = layers.Input(shape=(1,), dtype=tf.string, name='input_text')
vectorizer = layers.TextVectorization(max_tokens=max_vocab, output_sequence_length=max_len)
vectorizer.adapt(X_text_train) 
x_text = vectorizer(text_input)
x_text = layers.Embedding(input_dim=max_vocab, output_dim=32)(x_text)
x_text = layers.GlobalAveragePooling1D()(x_text)
x_text = layers.Dense(32, activation='relu')(x_text)

num_input = layers.Input(shape=(3,), name='input_numeric')
x_num = layers.Dense(32, activation='relu')(num_input)

merged = layers.Concatenate()([x_text, x_num])
attended = FluensyAttentionLayer()(merged)

x_shared = layers.Dense(128, activation='relu')(attended)
x_shared = layers.Dropout(0.2)(x_shared)
x_shared = layers.Dense(64, activation='relu')(x_shared)

# KEPALA TUNGGAL: Hanya Memprediksi Base Rate (Reels)
output_reels = layers.Dense(1, activation='linear', name='out_reels')(x_shared)

model_pricer = Model(inputs=[text_input, num_input], outputs=output_reels)

# =========================================================================
# 5. COMPILE & TRAINING
# =========================================================================
model_pricer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
    loss='huber',
    metrics=['mae']
)

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, mode='min')
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, mode='min', verbose=0)

print("🚀 Memulai Training The Pricer...")
history = model_pricer.fit(
    {'input_text': X_text_train, 'input_numeric': X_num_train},
    y_train,
    validation_data=(
        {'input_text': X_text_test, 'input_numeric': X_num_test},
        y_test
    ),
    epochs=100, 
    batch_size=16,
    callbacks=[early_stop, reduce_lr],
    verbose=0 
)

# =========================================================================
# 6. EVALUASI (Cek MAE Dicoding)
# =========================================================================
eval_results = model_pricer.evaluate(
    {'input_text': X_text_test, 'input_numeric': X_num_test},
    y_test,
    verbose=0
)

# Karena single output, Keras return array: [loss, mae]
mae_reels = eval_results[1]

print(f"\n=============================================")
print(f"🎯 MAE BASE RATE : {mae_reels:.4f} (Target Dicoding: < 0.02)")
print(f"=============================================")

# =========================================================================
# 7. EXPORT MODEL & SIMULASI THE MULTIPLIER ENGINE
# =========================================================================
model_pricer.save('pricer_model.keras')
print("\n💾 Model AI The Pricer disave ke 'pricer_model.keras'")

print("\n🔍 SIMULASI THE MULTIPLIER ENGINE (DI BACKEND):")
contoh_bio = tf.constant(["Gaya Hidup Perempuan, Fashion Wanita"], dtype=tf.string)
contoh_stats = scaler_X.transform([[1200000, 4.5, 24]]) # [Followers, ER, Umur]

pred_reels = model_pricer.predict({'input_text': contoh_bio, 'input_numeric': contoh_stats}, verbose=0)

# Mengembalikan skala angka
base_rate_raw = scaler_Y.inverse_transform(pred_reels)[0][0]
base_rate = max(300000, int(base_rate_raw)) 

print(f"\n[AI PREDICTION]")
print(f"-> Base Rate AI (Reels IG) : Rp {base_rate:,}")
print(f"\n[PENERAPAN DI DATABASE (tabel rate_cards)]")
print("Jika Platform = 'instagram':")
print(f"  - base_rate (Reels 100%) : Rp {base_rate:,}")
print(f"  - story_rate (25%)       : Rp {int(base_rate * 0.25):,}")
print(f"  - post_rate (70%)        : Rp {int(base_rate * 0.70):,}")
print(f"  - pp_rate (15%)          : Rp {int(base_rate * 0.15):,}")
print(f"  * addon_owning (50%)     : Rp {int(base_rate * 0.50):,}")
print(f"  * addon_boost (30%)      : Rp {int(base_rate * 0.30):,}")
print(f"  * addon_link (40%)       : Rp {int(base_rate * 0.40):,}")

print("\nJika Platform = 'tiktok':")
tiktok_base = int(base_rate * 1.20)
print(f"  - base_rate (Video 120%) : Rp {tiktok_base:,}")
print(f"  - story_rate (30%)       : Rp {int(base_rate * 0.30):,}")
print(f"  - post_rate (80%)        : Rp {int(base_rate * 0.80):,}")
print(f"  - pp_rate (20%)          : Rp {int(tiktok_base * 0.20):,}")
print(f"  * addon_owning (50%)     : Rp {int(tiktok_base * 0.50):,}")
print(f"  * addon_boost (30%)      : Rp {int(tiktok_base * 0.30):,}")
print(f"  * addon_link (50%)       : Rp {int(tiktok_base * 0.50):,}")

💸 [FLUENSY AI - THE PRICER] Membangun Model Prediksi Base Rate...

🚀 Memulai Training The Pricer...

🎯 MAE BASE RATE : 0.0123 (Target Dicoding: < 0.02)

💾 Model AI The Pricer disave ke 'pricer_model.keras'

🔍 SIMULASI THE MULTIPLIER ENGINE (DI BACKEND):


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(



[AI PREDICTION]
-> Base Rate AI (Reels IG) : Rp 5,837,796

[BACKEND MULTIPLIER (Blueprint Fluensy)]
A. INSTAGRAM RATE CARD
  - Reels (100%)       : Rp 5,837,796
  - Story (25%)        : Rp 1,459,449
  - Post Feed (70%)    : Rp 4,086,457
  - PP Spill (15%)     : Rp 875,669
  * Add-on Yellow Cart : +Rp 2,335,118

B. TIKTOK RATE CARD
  - Video (120%)       : Rp 7,005,355
  - Story (30%)        : Rp 1,751,338
  - Carousel (80%)     : Rp 4,670,236
  * Add-on Yellow Cart : +Rp 3,502,677
